---
title: Daily precipitation climatology and the annual march of the rain belt (GPM IMERG)
author: Harsha R. Hampapura
date: 2026-09-30
---

## Daily precipitation climatology: the annual march of the rain belt

[GPM IMERG Final (daily)](https://gdex.ucar.edu/datasets/d734000/) is a global, 0.1°
estimate of **daily** precipitation from NASA's satellite-and-gauge record. **GPM** —
the [Global Precipitation Measurement](https://climatedataguide.ucar.edu/climate-data/gpm-global-precipitation-measurement-mission)
mission — and **IMERG** (*Integrated Multi-satellitE Retrievals for GPM*) merge
microwave and infrared observations into one daily rain field; GDEX's copy spans
**June 2000 – September 2025**.

This notebook reproduces NASA's *IMERG Daily Climatology* animation from GDEX data. We

1. average every calendar day across all years into a **day-of-year climatology**,
2. smooth it with a **30-day trailing average** to remove the noise of isolated
   heavy-rain days, then
3. animate the 365-day cycle to watch the **Intertropical Convergence Zone (ITCZ)** —
   the equatorial band of heavy tropical rain — and the monsoons march north and south
   with the seasons.

Compared with the [monthly climatology notebook](./imerg_monthly_itcz.ipynb), the daily
record resolves seasonal transitions — monsoon onset, the ITCZ's migration — far more
smoothly.

:::{seealso} Original visualization
This workflow reproduces **"IMERG Daily Climatology"** — NASA's Scientific Visualization
Studio, [SVS #4759](https://svs.gsfc.nasa.gov/4759/) (released 15 October 2019).
Visualizer: Alex Kekesi (GST, Inc.); data visualizer: Horace Mitchell (NASA/GSFC);
scientists: George Huffman and Dalia B. Kirschbaum (NASA/GSFC); producers: Ryan
Fitzgibbons and Joy Ng (USRA). *Credit: NASA's Scientific Visualization Studio.* Our
version uses the GDEX IMERG daily record (2000–2025) rather than the 2001–2018 span in
the original.
:::

In [1]:
import os
import shutil
import calendar
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client, LocalCluster

## Open the dataset

IMERG daily is distributed as one NetCDF granule per day. GDEX adds a **kerchunk**
reference so a single `xr.open_dataset(..., engine="kerchunk")` call exposes the whole
~25-year daily record as one virtual `xarray.Dataset`.

The cell below auto-selects the access path: the POSIX reference where `/gdex/data` is
mounted (NCAR HPC), otherwise the OSDF reference streamed over the network. The OSDF
form's chunk URLs use the `osdf://` scheme, resolved by
[pelicanfs](https://github.com/PelicanPlatform/pelicanfs).

In [2]:
# POSIX reference where the GDEX data mount is present (NCAR HPC; its chunks live
# under /glade/campaign), otherwise the OSDF-routed reference over the network.
posix_ref = "/gdex/data/d734000/kerchunk/GPM_3IMERGDF_v07.json"
osdf_ref  = "https://data.gdex.ucar.edu/d734000/kerchunk/GPM_3IMERGDF_v07-remote-osdf.json"

kerchunk_ref = posix_ref if os.path.exists(posix_ref) else osdf_ref
print(f"Access: {'POSIX mount' if kerchunk_ref == posix_ref else 'OSDF (remote)'}")
print(kerchunk_ref)

Access: POSIX mount
/gdex/data/d734000/kerchunk/GPM_3IMERGDF_v07.json


## Set up a Dask cluster

Building the day-of-year climatology reads the **entire ~25-year daily record**, so we
farm the work out to a [Dask](https://docs.dask.org/) cluster. As in the other GDEX
examples, the cluster is chosen automatically by environment:

- **Casper / Derecho** (a PBS scheduler is present) → a `PBSCluster` spreads the read
  across scheduler-managed workers.
- **Laptop / CIRRUS Binder** (no PBS) → a `LocalCluster` uses the local cores.

In [3]:
def get_pbs_cluster(n_workers=20):
    """PBSCluster on NCAR HPC (Casper/Derecho)."""
    scratch = f"/glade/derecho/scratch/{os.environ.get('USER', '')}"
    cluster = PBSCluster(
        job_name        = "imerg-daily",
        cores           = 1,
        memory          = "8GiB",
        processes       = 1,
        local_directory = f"{scratch}/dask/spill/",
        log_directory   = f"{scratch}/dask/logs/",
        resource_spec   = "select=1:ncpus=1:mem=8GB",
        queue           = "casper",
        walltime        = "2:00:00",
        interface       = "ext",
    )
    cluster.scale(n_workers)
    return cluster


def get_local_cluster():
    """LocalCluster for a laptop or a CIRRUS Binder pod (no PBS scheduler)."""
    return LocalCluster()

:::{important} No edits needed
This cell **automatically selects the right Dask cluster** — a PBS cluster on NCAR HPC
(Casper/Derecho), or a `LocalCluster` on CIRRUS Binder or a laptop. **You don't need to
change anything;** the helper functions above show how each is built.
:::

In [4]:
on_hpc  = bool(shutil.which("qsub"))
cluster = get_pbs_cluster() if on_hpc else get_local_cluster()
client  = Client(cluster)
if on_hpc:
    client.wait_for_workers(n_workers=10)

print(f"Detected {'NCAR HPC -> PBSCluster' if on_hpc else 'no PBS -> LocalCluster'}")
cluster

Detected NCAR HPC -> PBSCluster


PBSCluster(5613f1b1, 'tcp://128.117.208.177:37663', workers=11, threads=11, memory=88.00 GiB)

## Day-of-year climatology

We average each calendar day (day-of-year 1–366) across all ~25 years to get the mean
annual cycle at daily resolution. Precipitation is already in **mm/day**, so no unit
conversion is needed; we mask the fill value by keeping non-negative rates.

:::{warning} This reads the entire daily record
The day-of-year mean touches **every one of the ~9,250 daily files (~240 GB)**, so
expect a substantial run — minutes on a PBS cluster, much longer streaming over OSDF.
We coarsen 0.1° → 0.2° up front to lighten the reduction and keep the animation small;
raise the coarsening factor (or subset the years) for a quicker first pass.
:::

In [ ]:
%%time
ds = xr.open_dataset(kerchunk_ref, engine="kerchunk", chunks={})
precip = ds["precipitation"].where(ds["precipitation"] >= 0)     # mm/day; mask fill
precip = precip.sel(time=slice("2020", "2025"))                  # most recent ~10 years

# Coarsen 0.1deg -> 0.2deg up front to lighten the reduction and the animation.
precip = precip.coarsen(lon=2, lat=2, boundary="trim").mean()

# Average each calendar day across those years -> the mean annual cycle.
doy_clim = precip.groupby("time.dayofyear").mean("time")         # (dayofyear, lon, lat)
doy_clim = doy_clim.transpose("dayofyear", "lat", "lon").compute()
doy_clim.attrs["units"] = "mm/day"
print("shape:", doy_clim.shape,
      "| dayofyear:", int(doy_clim.dayofyear[0]), "-", int(doy_clim.dayofyear[-1]))
doy_clim